# break_continue_pass
`break`, `continue`, and `pass` are small keywords with outsized control-flow impact. They determine when work stops, when work skips ahead, and when a block exists only as a placeholder. In production code, the danger is rarely syntax; it is hidden behavior.

## 1. Problem First
Why does this matter in real systems?
- A `break` can stop scanning too early and miss later critical records.
- A `continue` can skip validation or cleanup by mistake.
- A `pass` can leave a branch effectively unimplemented while the code still runs.

In [1]:
records = ["INFO", "WARN", "ERROR", "INFO"]
for record in records:
    if record == "ERROR":
        print("stopping on error")
        break
    print(record)

INFO
WARN
stopping on error


## 2. Minimal Concept
Core syntax:
- `break` exits the nearest loop immediately.
- `continue` skips the rest of the current iteration and moves to the next one.
- `pass` does nothing and is used as a placeholder.

## 3. Mental Model
How Python actually behaves
`break` changes loop termination, `continue` changes iteration flow, and `pass` changes nothing at runtime. These keywords are simple, but they alter which lines are reachable and which side effects happen. The real engineering skill is predicting what will be skipped.

In [2]:
numbers = [1, 2, 3, 4]
for number in numbers:
    if number % 2 == 0:
        continue
    print(number)

1
3


## 4. Internal Mechanics
These keywords compile into jump behavior in bytecode. `break` exits the loop block, `continue` jumps back to the loop check, and `pass` compiles into effectively no operation beyond keeping syntax valid.

In [3]:
import dis

def scan(values):
    for value in values:
        if value < 0:
            continue
        if value == 0:
            break
        pass
    return "done"

dis.dis(scan)
print(scan([3, 2, 0, 5]))

  3           0 RESUME                   0

  4           2 LOAD_FAST                0 (values)
              4 GET_ITER
        >>    6 FOR_ITER                15 (to 40)
             10 STORE_FAST               1 (value)

  5          12 LOAD_FAST                1 (value)
             14 LOAD_CONST               1 (0)
             16 COMPARE_OP               2 (<)
             20 POP_JUMP_IF_FALSE        1 (to 24)

  6          22 JUMP_BACKWARD            9 (to 6)

  7     >>   24 LOAD_FAST                1 (value)
             26 LOAD_CONST               1 (0)
             28 COMPARE_OP              40 (==)
             32 POP_JUMP_IF_FALSE        2 (to 38)

  8          34 POP_TOP

 10          36 RETURN_CONST             2 ('done')

  9     >>   38 JUMP_BACKWARD           17 (to 6)

  4     >>   40 END_FOR

 10          42 RETURN_CONST             2 ('done')
done


## 5. Edge Cases
Where it breaks:
- `continue` can skip logging, cleanup, or aggregation logic later in the loop.
- `break` only exits the nearest loop, not all nested loops.
- `pass` can hide unfinished logic in production paths.
- Loop control inside nested loops is often misunderstood.

In [4]:
processed = []
for raw in ["ok", "", "done"]:
    if not raw:
        continue
    processed.append(raw)
print(processed)

for row in [[1, 2], [3, 0], [4, 5]]:
    for value in row:
        if value == 0:
            break
    print("outer loop still continues")

['ok', 'done']
outer loop still continues
outer loop still continues
outer loop still continues


## 6. Performance Thinking
- `break` can save work by exiting early.
- `continue` can avoid unnecessary processing on bad or irrelevant records.
- Misused control keywords can also skip essential operations and create data-quality issues.
- The performance benefit matters only if correctness stays intact.

## 7. Real Use Case
A log scanner may skip malformed lines, stop on a fatal marker, and leave some branches unimplemented temporarily during development.

In [5]:
lines = ["INFO boot", "", "FATAL disk failed", "INFO later"]
for line in lines:
    if not line:
        continue
    if "FATAL" in line:
        print("alert and stop")
        break
    print("store", line)

store INFO boot
alert and stop


## 8. Anti-Pattern
What beginners do wrong:
- Use `continue` before cleanup code they still need.
- Assume `break` exits all loop levels.
- Leave `pass` in important branches and forget to replace it.

In [6]:
def handle_event(event):
    if event == "TODO":
        pass
    return "handled"

print(handle_event("TODO"))

handled


## 9. Interview Signals
Questions FAANG asks:
- What is the difference between `break` and `continue`?
- Why can `continue` create subtle bugs?
- What does `pass` do at runtime?
- How would you exit multiple nested loops cleanly?

## 10. Exercise (Non-trivial)
Design a batch processor that skips malformed records, stops immediately on poison-pill records, and guarantees metrics still update correctly. Explain where `continue`, `break`, and explicit helper functions are safer than a tangled loop body.

In [7]:
def process_batch(records):
    processed_count = 0
    for record in records:
        if not record:
            continue
        if record == "POISON":
            break
        processed_count += 1
    return processed_count

# Task:
# 1. Add separate malformed and poison-pill handling.
# 2. Make sure skipped records are still measurable.
# 3. Explain where loop control helps and where it hides logic.